# CUDA Lab 01 — CUDA Python, Thread Geometries, and Hardware Latency Pitfalls

**Student:** Seiitkhan Zhannur  
**Student ID:** 230109009  
**Date:** 08.10.2026

## Section 1: Environment Provisioning & Hardware Setup

In [ ]:
!nvidia-smi

Thu Oct  8 07:08:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from numba import cuda
import os

device = cuda.get_current_device()
name = device.name
if isinstance(name, bytes):
    name = name.decode('utf-8')

print(f"Device Name: {name}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64
Warp Size: 32


## Section 2: The Crossover Experiment

In [ ]:
import numpy as np
import time
from numba import cuda
# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out
# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
 # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
         d_out[idx] = d_in[idx] * 2.0

In [ ]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)
 # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _= double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Extrapolate or skip
 # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
 # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()
 # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0
 # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0
    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [ ]:
for N in [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]:
    cpu, total, kernel = benchmark_run(N)
    print(f"N={N:>10,} | CPU {cpu:10.3f} | GPU total {total:10.3f} | GPU kernel {kernel:10.3f}  (ms)")

N=       100 | CPU      0.071 | GPU total      0.722 | GPU kernel      0.093  (ms)
N=     1,000 | CPU      0.614 | GPU total      1.636 | GPU kernel      0.167  (ms)
N=    10,000 | CPU      6.107 | GPU total      0.690 | GPU kernel      0.092  (ms)
N=   100,000 | CPU     60.665 | GPU total      1.029 | GPU kernel      0.111  (ms)
N= 1,000,000 | CPU    571.673 | GPU total      6.709 | GPU kernel      0.180  (ms)
N= 5,000,000 | CPU        nan | GPU total     15.635 | GPU kernel      0.344  (ms)
N=10,000,000 | CPU        nan | GPU total     33.409 | GPU kernel      0.541  (ms)


## Section 3: Thread Geometry & Boundary Memory Corruption

### Task 3.1: Missing-guard kernel

In [ ]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
 # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0
# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")


Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


### Task 3.2: Thread coordinate inspector (N = 37, block size 8)

In [ ]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0
# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block # Exactly 5 blocks = 40 threads
log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()
print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40): # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


## Section 4: Race Condition (Parallel Reduction)

In [ ]:
# BUGGY NAIVE SUMMATION
@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
 # DATA RACE: Thousands of threads execute Read-Modify-Write at the exact same clock cycle!
        d_total[0] += d_arr[idx]
# CORRECT ATOMIC SUMMATION
@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
 # HARDWARE ATOMIC LOCK: Hardware serializ
        cuda.atomic.add(d_total, 0, d_arr[idx])

In [ ]:
N = 50_000
d_arr = cuda.to_device(np.ones(N, dtype=np.float32))
tpb = 256
bpg = (N + tpb - 1) // tpb

for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_arr, d_total)
    cuda.synchronize()
    out = d_total.copy_to_host()[0]
    print(f"Trial {trial}: {out} | lost: {N - out}")

Trial 1: 2.0 | lost: 49998.0
Trial 2: 2.0 | lost: 49998.0
Trial 3: 2.0 | lost: 49998.0
Trial 4: 2.0 | lost: 49998.0
Trial 5: 2.0 | lost: 49998.0


In [ ]:
def time_kernel(kernel):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    t0 = time.perf_counter()
    kernel[bpg, tpb](d_arr, d_total)
    cuda.synchronize()
    return (time.perf_counter() - t0) * 1000, d_total.copy_to_host()[0]

time_kernel(atomic_sum_kernel)   # first run only compiles, ignore it
ms_naive, _ = time_kernel(naive_sum_kernel)
ms_atomic, total = time_kernel(atomic_sum_kernel)
print(f"Atomic sum: {total} | naive {ms_naive:.3f} ms | atomic {ms_atomic:.3f} ms")

Atomic sum: 50000.0 | naive 0.190 ms | atomic 0.336 ms


## Section 5: 2D Grid Processing (Radial Vignette)

In [1]:
import math
import numpy as np
from numba import cuda
@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
 # Retrieve 2D spatial coordinates directly from the grid
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

# 2D EXECUTION CONFIGURATION
WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)
# 2D Blocks (16x16 = 256 threads per block)
threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)
radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512,512]:.3f}")


Generated 2D matrix shape: (1024, 1024). Center value: 1.000


In [2]:
print(f"Corner (0, 0) value: {result_canvas[0, 0]:.3f}")

Corner (0, 0) value: 0.000


## Section 6: Verification Checksum

In [3]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    # Include GPU name
    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode('utf-8')
    h.update(gpu_name)
    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230109009"
CROSSOVER_N = 10000  # Q1: first N where GPU Total < CPU Time
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: C4AD983365469756


'C4AD983365469756'